<a href="https://colab.research.google.com/github/anjaruzica/final_project/blob/main/hangover_hypothesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Hangover Hypothesis

An Econometric Investigation of Alcohol's Impact on Academic Performance

**Authors:** Anja, Marina and Sergio

**Course:** Economic Modeling & Simulation  
**Professor:** Bogdan-Alexandru Ratiu  
**Date:** 22 May 2026

**Research question:** Does drinking actually hurt your grades, or are professors just trying to scare us?

## 1. Dataset Description

We used the **UCI Student Performance dataset** (Cortez & Silva, 2008), publicly available at the UCI Machine Learning Repository. It contains data from **395 students** in two Portuguese secondary schools, collected through surveys and school records.

The dataset has been cited in multiple academic papers about education and data mining.

Why we chose this datasetWe picked this one because it lets us answer something every student has wondered about: how much does drinking actually affect your grades? It's real data (not simulated), well-documented, and has both continuous and binary outcomes which means we can use both linear and logistic regression on it. It also satisfies the project requirements: 395 rows (well above 200) and 33 features (well above 4).

Key variables for our analysis
 **Dalc** Weekday alcohol consumption (1 = very low, 5 = very high) **Walc** Weekend alcohol consumption (1 = very low, 5 = very high)  **studytime** Weekly study time (1 = <2h, 4 = >10h)
 **failures** Number of past class failures (0–3) **absences**

Number of school absences **goout**
How often the student goes out with friends (1 = very low, 5 = very high) **G3**

Final grade in math (0–20, the Portuguese grading system) **passed**
Binary variable we create later: G3 ≥ 10 (passing grade in Portugal)


**Source:** Cortez, P. and Silva, A. (2008). *Using Data Mining to Predict Secondary School Student Performance*. UCI Machine Learning Repository.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, r2_score, mean_squared_error
)

# style for the plots
plt.style.use("default")
np.random.seed(42)

In [ ]:
# Load the dataset
# After uploading student-mat.csv in Colab (left sidebar -> files -> upload):
df = pd.read_csv("student-mat.csv")

print(f"Shape of the dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()

##

2. Data Cleaning and PreparationBefore doing any analysis we need to check the data is clean: any missing values, any weird outliers, any categorical variables we need to encode for the regression models?

In [ ]:
# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values in any column!")

print(f"\nTotal missing values in the dataset: {df.isnull().sum().sum()}")
print(f"\nData types:")
print(df.dtypes.value_counts())

**Missing values check:** The dataset has zero missing values across all 33 columns. This is one of the reasons it's used so much in academic teaching.
Cortez & Silva cleaned it well before publishing.Most columns are integers (numerical features like alcohol level, grades, absences) and the rest are strings (categorical features like sex, school, etc.).

In [ ]:
# Check for outliers in the numeric columns we care about
print("Statistics for key numeric variables:")
print(df[['Dalc', 'Walc', 'studytime', 'absences', 'failures', 'G3']].describe().round(2))

print(f"\nStudents with final grade G3 = 0: {(df['G3'] == 0).sum()}")
print("(These are likely students who dropped out or did not take the final exam)")

print(f"\nStudents with more than 30 absences: {(df['absences'] > 30).sum()}")
print(f"Max absences: {df['absences'].max()}")

**Outlier check:** There are 38 students with G3 = 0, which are likely students who dropped out or did not show up to the final exam. We keep them in the dataset because dropping out *is* a real outcome we want to predict removing them would bias the model toward only the students who showed up.A few students have extremely high absences (max 75), but absences is a meaningful predictor by itself, so we keep these too. The data is realistic and we don't need to drop anything.

In [ ]:
# Feature engineering: create new variables and encode categoricals

# 1. Binary target variable: did the student pass? (G3 >= 10 in Portugal)
df['passed'] = (df['G3'] >= 10).astype(int)

# 2. Combined alcohol consumption (weekday + weekend)
df['total_alcohol'] = df['Dalc'] + df['Walc']

# 3. Categorical drinking level for grouped analysis
def drinking_category(total):
    if total <= 3:
        return "Low"
    elif total <= 6:
        return "Moderate"
    else:
        return "Heavy"

df['drinking_category'] = df['total_alcohol'].apply(drinking_category)

# 4. Encode sex as a binary numeric variable (needed for regression)
df['sex_male'] = (df['sex'] == 'M').astype(int)

# show what we created
print("New columns we created:")
print(df[['G3', 'passed', 'total_alcohol', 'drinking_category', 'sex_male']].head(10))

**Feature engineering:** We added four new variables:1. **`passed`** Binary target for logistic regression: 1 if the student got a passing grade (G3 ≥ 10 in the Portuguese system), 0 otherwise.2. **`total_alcohol`** Sum of weekday + weekend drinking. Captures overall alcohol consumption in a single feature.3. **`drinking_category`** Categorical bin of total alcohol into Low / Moderate / Heavy. Useful for groupby visualizations.4. **`sex_male`**  Numerical encoding of the categorical `sex` column. Most ML models can't take strings as input, so we convert F → 0 and M → 1.

## 3. Exploratory Data AnalysisNow that the data is clean and we have our engineered features, let's look at what's actually going on with summary statistics, group-by analysis, and visualizations.

In [ ]:
# Summary statistics for our key variables
print("Summary statistics:")
print(df[['Dalc', 'Walc', 'total_alcohol', 'studytime', 'absences', 'G3']].describe().round(2))

print(f"\nOverall pass rate: {df['passed'].mean() * 100:.1f}%")
print(f"Average final grade (G3): {df['G3'].mean():.2f} / 20")

# Group by drinking category
print("\nPass rate and average grade by drinking category:")
print(df.groupby('drinking_category').agg(
    n_students=('G3', 'size'),
    avg_grade=('G3', 'mean'),
    pass_rate=('passed', 'mean')
).round(3))

### Visualization 1: Distribution of final grades

In [ ]:
# Visualization 1: histogram of final grades
plt.figure(figsize=(12, 6))
plt.hist(df['G3'], bins=21, color='purple', edgecolor='black', alpha=0.75)
plt.axvline(x=10, color='red', linestyle='--', linewidth=2, label='Passing threshold (G3 = 10)')
plt.title('Distribution of Final Math Grades (G3)', fontsize=14)
plt.xlabel('Final grade (0-20)')
plt.ylabel('Number of students')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**What this shows:** The grade distribution is roughly bell-shaped between 5 and 18, with a big spike at 0 (the 38 dropout students we identified earlier). The red dashed line at G3 = 10 is the passing threshold in the Portuguese system. Most students cluster between 8 and 14, with the median right around the passing line meaning math is genuinely difficult and many students are borderline.

### Visualization 2: Grades by drinking category

In [ ]:
# Visualization 2: box plot of grades by drinking category
plt.figure(figsize=(10, 6))

# order matters for the visualization
order = ['Low', 'Moderate', 'Heavy']
data_to_plot = [df.loc[df['drinking_category'] == cat, 'G3'] for cat in order]

bp = plt.boxplot(data_to_plot, labels=order, patch_artist=True)

# color the boxes from green to red
colors = ['lightgreen', 'gold', 'salmon']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

plt.axhline(y=10, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Passing threshold')
plt.title('Final Grade Distribution by Drinking Category', fontsize=14)
plt.xlabel('Drinking category (total alcohol score)')
plt.ylabel('Final grade (G3)')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

**What this shows:** The medians across drinking categories are surprisingly close. Light drinkers have a slightly higher median grade than moderate and heavy drinkers, but the difference is much smaller than we expected. There's a lot of overlap between all three groups — meaning alcohol alone doesn't separate the students well. The interquartile ranges (the boxes) also overlap heavily, which already hints that alcohol is not as strong a predictor as the popular narrative claims.

### Visualization 3: Weekend alcohol vs final grade (with jitter)

In [ ]:
# Visualization 3: scatter plot of Walc vs G3
# since Walc is discrete (1-5) we add jitter so points don't all stack
plt.figure(figsize=(10, 6))

jitter_x = df['Walc'] + np.random.uniform(-0.15, 0.15, size=len(df))
jitter_y = df['G3'] + np.random.uniform(-0.15, 0.15, size=len(df))

# color by pass/fail
passed_mask = df['passed'] == 1
plt.scatter(jitter_x[passed_mask], jitter_y[passed_mask],
            alpha=0.5, color='green', label='Passed (G3 >= 10)', s=30)
plt.scatter(jitter_x[~passed_mask], jitter_y[~passed_mask],
            alpha=0.5, color='red', label='Failed (G3 < 10)', s=30)

plt.axhline(y=10, color='gray', linestyle='--', linewidth=1)
plt.title('Weekend Alcohol Consumption vs Final Grade', fontsize=14)
plt.xlabel('Weekend alcohol consumption (Walc, 1=very low, 5=very high)')
plt.ylabel('Final grade (G3)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**What this shows:** Pretty mixed picture. We added small jitter to the points so they're visible — without it they'd all stack on the integer values of Walc (1-5). What we can see is that **green (passed) and red (failed) students appear at every level of weekend drinking**. Even students who drink heavily on weekends (Walc = 5) still pass at a similar rate to lighter drinkers. Visually, there's no obvious downward trend, which is consistent with the very low correlation (-0.058) we computed earlier.

### Visualization 4: Pass rate by weekday drinking level

In [ ]:
# Visualization 4: bar chart of pass rate by Dalc level
pass_by_dalc = df.groupby('Dalc').agg(
    pass_rate=('passed', 'mean'),
    n_students=('passed', 'size')
).reset_index()

plt.figure(figsize=(10, 6))
bars = plt.bar(pass_by_dalc['Dalc'], pass_by_dalc['pass_rate'] * 100,
               color=['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c'],
               edgecolor='black')

# add labels on top of each bar
for bar, n in zip(bars, pass_by_dalc['n_students']):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height + 1,
             f'{height:.1f}%\n(n={n})', ha='center', va='bottom', fontsize=10)

plt.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')
plt.title('Pass Rate by Weekday Alcohol Level (Dalc)', fontsize=14)
plt.xlabel('Weekday alcohol consumption (Dalc, 1=very low, 5=very high)')
plt.ylabel('Pass rate (%)')
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

**What this shows:**

Pass rate generally declines as weekday drinking increases from ~70% at Dalc=1 down to ~44% at Dalc=4.

 This is the trend we expected! However, the Dalc=5 group (very heavy weekday drinkers) actually has the highest pass rate. Looking at the sample sizes printed on each bar explains why: only 9 students have Dalc=5, so that result is statistically unreliable.

 The honest takeaway: there's a *visible* negative trend in pass rate as weekday alcohol goes from 1 to 4, but the effect at the extreme levels is noisy due to small sample sizes.

### Visualization 5: Correlation between key variables

In [ ]:
# Visualization 5: correlation heatmap (using matplotlib only, no seaborn)
corr_cols = ['Dalc', 'Walc', 'studytime', 'absences', 'failures', 'goout', 'G3']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# add labels
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right')
ax.set_yticklabels(corr_cols)

# annotate the cells with correlation values
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        value = corr_matrix.iloc[i, j]
        color = 'white' if abs(value) > 0.5 else 'black'
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im, ax=ax, label='Correlation')
plt.title('Correlation Matrix of Key Variables', fontsize=14)
plt.tight_layout()
plt.show()

**What this shows:** This is the most informative plot for understanding what actually matters. Looking at the bottom row (G3 vs everything else):- **G3 ↔ failures**: -0.36 (strongest negative correlation). Past failures predict future failures.- **G3 ↔ goout**: -0.13 (going out a lot has a small negative effect)- **G3 ↔ Dalc**: -0.05 and **G3 ↔ Walc**: -0.05 — alcohol has essentially no direct linear correlation with grades- **G3 ↔ studytime**: +0.10 (positive but weaker than you'd hope)- **G3 ↔ absences**: +0.03 (basically zero!)The big surprise is that **alcohol consumption barely correlates with final grades**. The strongest predictor of academic failure is past academic failure, not drinking habits.

## 4. Object-Oriented Programming: the Student classTo keep the case-study analysis clean and readable, we wrap each student in a `Student` class. This is the same OOP pattern we used in class for Point → ColorPoint → AdvancedPoint: an `__init__` constructor with validation, `@property` getters, a `__str__` method, and class variables for thresholds.

In [ ]:
class Student:
    """
    A class representing one student in the dataset.
    Same OOP pattern we used for Point / ColorPoint / AdvancedPoint.
    """

    # class variable: the threshold below which the student fails
    PASSING_GRADE = 10

    # combined alcohol score above this is "heavy" drinking
    HEAVY_DRINKING_THRESHOLD = 6

    def __init__(self, dalc, walc, studytime, absences, failures, goout, g3):
        """
        :param dalc: weekday alcohol consumption (1=very low, 5=very high)
        :param walc: weekend alcohol consumption (1=very low, 5=very high)
        :param studytime: weekly study time (1=<2h, 4=>10h)
        :param absences: number of school absences
        :param failures: number of past class failures (0-3)
        :param goout: how often the student goes out (1=very low, 5=very high)
        :param g3: final grade (0-20)
        """
        if dalc < 1 or dalc > 5:
            raise ValueError("Dalc must be between 1 and 5")
        if walc < 1 or walc > 5:
            raise ValueError("Walc must be between 1 and 5")
        self._dalc = dalc
        self._walc = walc
        self._studytime = studytime
        self._absences = absences
        self._failures = failures
        self._goout = goout
        self._g3 = g3

    @property
    def dalc(self):
        return self._dalc

    @property
    def walc(self):
        return self._walc

    @property
    def total_alcohol(self):
        """Combined weekday + weekend drinking score."""
        return self._dalc + self._walc

    @property
    def is_heavy_drinker(self):
        return self.total_alcohol >= self.HEAVY_DRINKING_THRESHOLD

    @property
    def passed(self):
        """True if the student passed (G3 >= 10)."""
        return self._g3 >= self.PASSING_GRADE

    def __str__(self):
        return (f"Student<Dalc={self._dalc}, Walc={self._walc}, "
                f"studytime={self._studytime}, absences={self._absences}, "
                f"failures={self._failures}, goout={self._goout}, "
                f"G3={self._g3}, passed={self.passed}>")

    def __repr__(self):
        return self.__str__()


# test the class
test_student = Student(dalc=2, walc=4, studytime=2, absences=5, failures=0, goout=4, g3=12)
print(test_student)
print(f"Heavy drinker? {test_student.is_heavy_drinker}")
print(f"Total alcohol score: {test_student.total_alcohol}")

## 5. ModelingWe use two regression models on the same set of features:- **Logistic Regression** to predict the binary outcome (pass / fail)- **Linear Regression** to predict the actual final grade (G3)The features used for both: `Dalc`, `Walc`, `studytime`, `absences`, `failures`, `goout`.

### 5.1 Logistic Regression — predicting pass/fail

In [ ]:
# Logistic Regression: predict pass/fail
features = ['Dalc', 'Walc', 'studytime', 'absences', 'failures', 'goout']
X = df[features].values
y = df['passed'].values

# Train/test split (80/20) with stratification to preserve the pass/fail ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# fit the model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# print coefficients
print("Logistic Regression coefficients:")
for name, coef in zip(features, log_model.coef_[0]):
    sign = "+" if coef >= 0 else ""
    print(f"  {name:12s} {sign}{coef:.4f}")
print(f"  {'intercept':12s} {log_model.intercept_[0]:+.4f}")

In [ ]:
# Evaluate on the held-out test set
y_pred = log_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=== Logistic Regression Performance (test set) ===")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print()
print("Confusion matrix:")
print(f"                  predicted FAIL  predicted PASS")
print(f"  actual FAIL          {cm[0][0]:3d}              {cm[0][1]:3d}")
print(f"  actual PASS          {cm[1][0]:3d}              {cm[1][1]:3d}")

# visualize the confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Purples')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted FAIL', 'Predicted PASS'])
ax.set_yticklabels(['Actual FAIL', 'Actual PASS'])

for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color=color, fontsize=20, fontweight='bold')

plt.title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Logistic Regression results interpretation:**

 **Accuracy 65.8%** : the model predicts pass/fail correctly about two-thirds of the time. Considering the overall pass rate is 67%, the model is barely doing better than just guessing "pass" for everyone. That's important to acknowledge honestly.

 **Recall 84.9%** : when a student actually passed, the model caught it 85% of the time. The model is good at finding passers.

  **Precision 70.3%** : when the model predicts "pass," it's right 70% of the time.
  
  
  What the coefficients tell us:
  
   **`- failures`** has the largest negative coefficient (-1.17). Each past failure dramatically reduces the probability of passing. This is the single strongest predictor.- **`goout`** has a notable negative coefficient (-0.40). Going out a lot hurts your chances.- **`Dalc`** (weekday alcohol) is slightly negative (-0.16) small effect but in the expected direction.- **`Walc`** (weekend alcohol) is actually slightly *positive* (+0.36). Weekend drinking alone is not predictive of failing  possibly because it reflects social integration rather than dysfunction.- **`studytime`** is positive (+0.06) but smaller than expected.

### 5.2 Linear Regression — predicting the actual grade

In [ ]:
# Linear Regression: predict the actual final grade (G3)
y_grades = df['G3'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y_grades, test_size=0.2, random_state=42
)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred = lin_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Linear Regression Performance (test set) ===")
print(f"R² score: {r2:.3f}")
print(f"RMSE:     {rmse:.3f} grade points")
print()
print("Linear Regression coefficients:")
for name, coef in zip(features, lin_model.coef_):
    sign = "+" if coef >= 0 else ""
    print(f"  {name:12s} {sign}{coef:.4f}")
print(f"  {'intercept':12s} {lin_model.intercept_:+.4f}")

# visualize predicted vs actual
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.5, color='purple')
plt.plot([0, 20], [0, 20], 'r--', label='Perfect prediction')
plt.xlabel('Actual final grade (G3)')
plt.ylabel('Predicted final grade')
plt.title(f'Predicted vs Actual Grades (R² = {r2:.3f})')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(-1, 21)
plt.ylim(-1, 21)
plt.show()

**Linear Regression results — interpretation:**- **R² = 0.06**

 our features explain only 6% of the variance in final grades. That's low, and the predicted-vs-actual plot shows why: predictions cluster around the average grade and don't capture the spread.
 - **RMSE = 4.39**,  on average our predictions are off by about 4.4 grade points (out of 20). Not great.

 **Why is the fit so weak?** Because final grades in math depend on a lot of things our six features don't capture: aptitude, teacher quality, test difficulty, sleep, mental health, family situation, etc. Six lifestyle variables can't predict an entire academic outcome.


**What the coefficients say (even if the overall fit is weak):**- **`failures`** coefficient of -2.24, each past failure predicts a 2.24-point drop in the final grade. By far the strongest effect.
- **`studytime`** coefficient of +0.53, each additional study-time bracket adds about half a grade point.
- **`Dalc`** coefficient is basically 0 (-0.02), weekday alcohol has almost no linear effect on grade.
- **`Walc`** coefficient is +0.12, again, slightly positive, very small.

## 6. Case Study and Marginal AnalysisUsing the OOP class we built, we now analyze a specific "case study" student — someone with poor habits — and ask the models: what's their chance of passing, and what would happen if they drank one more beer?

In [ ]:
# Create a case study using our Student class
case = Student(dalc=3, walc=4, studytime=1, absences=12, failures=1, goout=5, g3=0)
print("Case study student:")
print(case)
print(f"Heavy drinker? {case.is_heavy_drinker}")
print()

# Predict their probability of passing
case_features = np.array([[case.dalc, case.walc, 1, 12, 1, 5]])
prob_pass = log_model.predict_proba(case_features)[0][1]
predicted_grade = lin_model.predict(case_features)[0]

print(f"Predicted probability of passing: {prob_pass * 100:.2f}%")
print(f"Predicted final grade (G3):       {predicted_grade:.2f} / 20")

In [ ]:
# Marginal analysis: what's the cost of one more weekend beer?
case_plus_features = np.array([[case.dalc, case.walc + 1, 1, 12, 1, 5]])
grade_plus = lin_model.predict(case_plus_features)[0]
prob_plus = log_model.predict_proba(case_plus_features)[0][1]

print(f"=== Marginal cost of one more weekend drink ===")
print(f"At Walc = {case.walc}: P(pass) = {prob_pass*100:.2f}%, predicted G3 = {predicted_grade:.2f}")
print(f"At Walc = {case.walc + 1}: P(pass) = {prob_plus*100:.2f}%, predicted G3 = {grade_plus:.2f}")
print(f"Change in P(pass):  {(prob_plus - prob_pass)*100:+.2f} percentage points")
print(f"Change in grade:    {grade_plus - predicted_grade:+.3f} grade points")

In [ ]:
# Failure threshold: at what total alcohol level does P(pass) drop below 50%?
def find_failure_threshold(studytime=1, absences=12, failures=1, goout=5):
    """Find the lowest alcohol level (Walc=Dalc) at which P(pass) drops below 50%."""
    for level in range(1, 6):
        features_array = np.array([[level, level, studytime, absences, failures, goout]])
        prob = log_model.predict_proba(features_array)[0][1]
        if prob < 0.5:
            return level, prob
    return None, None

threshold, prob_at_threshold = find_failure_threshold()
print("=== Failure Threshold ===")
if threshold is not None:
    print(f"At alcohol level {threshold} (both Dalc and Walc), P(pass) drops to {prob_at_threshold*100:.2f}%")
else:
    print("Even at maximum alcohol, P(pass) stays above 50% for these other features.")

## 7. Conclusions, Insights, and Limitations

 What we found

 1. Alcohol's effect on grades is much weaker than we expected.

The correlation between total alcohol consumption and final grade is only **-0.058**  essentially zero. The R² of our linear regression is 0.06, meaning lifestyle factors alone explain only 6% of the variance in grades.

 **2. Past failures matter way more than current drinking.

 The single strongest predictor in both models is the `failures` variable. Each past class failure predicts a 2.24-point grade drop and large reduction in pass probability. This makes intuitive sense — students who failed once are more likely to fail again.

 **3. Social behavior matters more than alcohol per se.**  
 The `goout` (how often you go out with friends) variable has a stronger negative coefficient than either Dalc or Walc. This suggests that *being out and not studying* matters more than the alcohol itself.

 **4. Weekend drinking has no measurable negative effect.**
  Our Walc coefficient is actually slightly *positive* in both models. One interpretation: weekend drinking reflects normal social integration and doesn't directly interfere with weekday studying. Drinking on weekdays (Dalc) is more harmful, though still small.
  
  ### Limitations
  
  **Correlation is not causation.** Even if drinking and grades correlated strongly, we couldn't say drinking *caused* low grades — both could be caused by something else (e.g., personality, family stability, mental health).
  
  **Single school, single subject.** The data is from two Portuguese schools and only math grades. Other subjects or countries might behave differently.
  
  **Self-reported alcohol consumption.** Students might under-report their drinking. The 1-5 scale is also coarse, there's no distinction between 6 beers and 12 beers per weekend.
  
   **The R² of our linear model is low (0.06).** Six lifestyle variables can't predict a whole academic outcome. Important missing features include teacher quality, test difficulty, mental health, sleep, and academic aptitude.- **The 38 students with G3=0** are likely dropouts, which makes the distribution bimodal and harder to model with a single linear equation.
   
   · Final answer to our research question :  **Does drinking actually hurt your grades?** Based on this dataset: **slightly, but not nearly as much as the cultural narrative suggests.** Weekday drinking has a small negative effect on pass probability, weekend drinking has none. What actually predicts failure is having failed before and spending a lot of time going out with friends, the *behavior pattern around* drinking matters more than the alcohol itself.So: the Hangover Hypothesis is **partially supported but mostly overstated**. The real culprit isn't the beer, it's the missed Tuesday lecture afterward.